# Maria Sacha Paper Pipeline Reproduction (Integrated TVBToolkit)

This notebook reproduces the key analysis blocks from
`paper_pipeline_hub/whole_pipeline.ipynb` using **integrated TVBToolkit modules**.

Integrated module mapping:
- whole-brain conditions and batch runs: `tvbtoolkit.workflows.presets` + `tvbtoolkit.workflows.experiments`
- PCI summary: `tvbtoolkit.complexity` + `tvbtoolkit.visualization`
- BOLD FC-SC coupling: `tvbtoolkit.bold` + `tvbtoolkit.whole_brain.analysis`
- survival heatmaps: `tvbtoolkit.analysis.dynamics`
- single-region helper functions: `tvbtoolkit.single_region.analysis`

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from tvb.simulator import lab

from tvbtoolkit import (
    WholeBrainConfig,
    OutputConfig,
    run_condition_batch,
    maria_sacha_nature_conditions,
    plot_example_timeseries,
    plot_metric_summary,
    fcsc_seedwise_from_saved_batch,
    load_survival_arrays,
    plot_survival_heatmap,
    bin_array,
    input_rate,
    calculate_psd_fmax,
)
from tvbtoolkit.visualization import set_publication_style

set_publication_style()

In [ ]:
PAPER_ROOT = Path('/Users/borjan/CNRS/projects/paper_pipeline_hub')
OUT_ROOT = Path('outputs') / 'repro_maria_sacha_nature'

RUN_CFG = {
    'quick': True,
    'simulation_length_ms': 12000.0,
    'cut_transient_ms': 4000.0,
    'seeds': [0] if True else [0,1,2],
    'n_jobs': 1,
    'use_processes': False,
    'monitor_mode_default': 'temporal_average',
    'temporal_average_period_ms': 1.0,
    'tr_ms': 500.0,
}

output = OutputConfig(root=OUT_ROOT)
fig_dir = output.figures_dir / 'paper_repro'
fig_dir.mkdir(parents=True, exist_ok=True)

print('paper root:', PAPER_ROOT)
print('output root:', output.root.resolve())
print('fig dir:', fig_dir.resolve())

## 1) Whole-brain paper-style conditions (wake, nmda, gaba, sleep)

Parameterization is integrated via `maria_sacha_nature_conditions()` and
paper-aligned transfer-function coefficients from the original notebook.

In [ ]:
P_e = [-0.05017034, 0.00451531, -0.00794377, -0.00208418, -0.00054697,
       0.00194753, 0.00274079, 0.00341614, -0.01066769, -0.01156433]
P_i = [-0.05184978, 0.0061593, -0.01403522, 0.00166511, -0.0020559,
       0.00656668, 0.00171829, 0.00318432, -0.04516385, -0.03112775]

conn_76 = PAPER_ROOT / 'TVB' / 'tvb_model_reference' / 'data' / 'connectivity' / 'connectivity_76.zip'
if not conn_76.exists():
    raise FileNotFoundError(f'Missing connectivity file: {conn_76}')

base_cfg = WholeBrainConfig(
    model_family='adex_zerlaut',
    zerlaut_matteo=False,
    zerlaut_gk_gna=False,
    zerlaut_order=1,
    stochastic_integrator=True,
    simulation_length_ms=RUN_CFG['simulation_length_ms'],
    dt_ms=0.1,
    conduction_speed=4.0,
    coupling_strength=0.3,
    connectivity_zip=str(conn_76),
    monitor_mode='temporal_average',
    temporal_average_period_ms=RUN_CFG['temporal_average_period_ms'],
    parameter_overrides={
        'parameter_model': {
            'E_L_e': -64.0,
            'E_L_i': -65.0,
            'P_e': P_e,
            'P_i': P_i,
        }
    },
)

conditions = maria_sacha_nature_conditions()
[c.name for c in conditions]

In [ ]:
metrics_by_condition = run_condition_batch(
    base_cfg=base_cfg,
    conditions=conditions,
    seeds=RUN_CFG['seeds'],
    output=output,
    post_stim_window=300,
    save_timeseries=True,
    n_jobs=RUN_CFG['n_jobs'],
    use_processes=RUN_CFG['use_processes'],
    show_progress=True,
    monitor_mode_default=RUN_CFG['monitor_mode_default'],
    temporal_average_period_ms=RUN_CFG['temporal_average_period_ms'],
)

list(metrics_by_condition.keys())

In [ ]:
fig1 = plot_example_timeseries(
    metrics_by_condition,
    max_regions=8,
    trim_first_ms=RUN_CFG['cut_transient_ms'],
    save_path=fig_dir / '01_paper_style_timeseries.svg',
)
fig1.savefig(fig_dir / '01_paper_style_timeseries.pdf', bbox_inches='tight')

fig2 = plot_metric_summary(
    metrics_by_condition,
    save_path=fig_dir / '02_pci_lzc_summary.svg',
)
fig2.savefig(fig_dir / '02_pci_lzc_summary.pdf', bbox_inches='tight')
plt.show()

## 2) FC-SC coupling on BOLD-like signals (seedwise)

This uses integrated function `fcsc_seedwise_from_saved_batch(...)`.

In [ ]:
conn = lab.connectivity.Connectivity().from_file(str(conn_76))
SC = np.asarray(conn.weights, dtype=float)
np.fill_diagonal(SC, 0.0)

fcsc = fcsc_seedwise_from_saved_batch(
    output,
    conditions=[c.name for c in conditions],
    seeds=RUN_CFG['seeds'],
    structural_connectivity=SC,
    cut_transient_ms=RUN_CFG['cut_transient_ms'],
    tr_ms=RUN_CFG['tr_ms'],
)

for cond, vals in fcsc.items():
    legacy = vals['legacy_r_signed_full']
    masked = vals['masked_r_abs_upper']
    print(
        f"{cond:>6}: legacy={np.nanmean(legacy):.3f}±{np.nanstd(legacy):.3f}, "
        f"masked|FC|={np.nanmean(masked):.3f}±{np.nanstd(masked):.3f}"
    )

In [ ]:
labels = [c.name for c in conditions]
legacy_means = [np.nanmean(fcsc[l]['legacy_r_signed_full']) for l in labels]
masked_means = [np.nanmean(fcsc[l]['masked_r_abs_upper']) for l in labels]

x = np.arange(len(labels))
width = 0.38
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.bar(x - width/2, legacy_means, width=width, label='Legacy signed FC-SC')
ax.bar(x + width/2, masked_means, width=width, label='Masked |FC|-SC')
ax.set_xticks(x, [l.upper() for l in labels])
ax.set_ylabel('Coupling (r)')
ax.set_title('Paper-style FC-SC coupling by condition')
ax.grid(alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(fig_dir / '03_fcsc_seedwise_summary.svg', bbox_inches='tight')
fig.savefig(fig_dir / '03_fcsc_seedwise_summary.pdf', bbox_inches='tight')
plt.show()

## 3) Dynamical-analysis survival heatmaps (precomputed arrays)

Uses integrated functions:
- `load_survival_arrays(...)`
- `plot_survival_heatmap(...)`

In [ ]:
mean_e, taus_e, bthr_e, tau_v_e, bvals_e = load_survival_arrays(
    load='tau_e',
    precalc=True,
    paper_repo_root=PAPER_ROOT,
)
fig_e = plot_survival_heatmap(
    mean_e,
    taus_e,
    bthr_e,
    tau_v_e,
    bvals_e,
    load='tau_e',
    save_path=fig_dir / '04_survival_heatmap_tau_e.svg',
)
fig_e.savefig(fig_dir / '04_survival_heatmap_tau_e.pdf', bbox_inches='tight')

mean_i, taus_i, bthr_i, tau_v_i, bvals_i = load_survival_arrays(
    load='tau_i',
    precalc=True,
    paper_repo_root=PAPER_ROOT,
)
fig_i = plot_survival_heatmap(
    mean_i,
    taus_i,
    bthr_i,
    tau_v_i,
    bvals_i,
    load='tau_i',
    save_path=fig_dir / '05_survival_heatmap_tau_i.svg',
)
fig_i.savefig(fig_dir / '05_survival_heatmap_tau_i.pdf', bbox_inches='tight')
plt.show()

## 4) Single-region analysis utility parity check

Demonstrates that integrated single-region helper functions are available in
`tvbtoolkit.single_region` namespace.

In [ ]:
t_ms = np.arange(0.0, 4000.0, 1.0)
exc = 12.0 + 3.0*np.sin(2*np.pi*8*(t_ms/1000.0))
inh = 8.0 + 2.0*np.sin(2*np.pi*8*(t_ms/1000.0) + 0.4)

inp = input_rate(t_ms, t1_exc=1000.0, tau1_exc=40.0, tau2_exc=80.0, ampl_exc=5.0, plateau=200.0)
exc_drive = exc + inp

tb = bin_array(t_ms, 5.0, t_ms)
exc_b = bin_array(exc_drive, 5.0, t_ms)
inh_b = bin_array(inh, 5.0, t_ms)

fmax, frq, pexc, pinh = calculate_psd_fmax(exc_b, inh_b, tb)
print('Peak frequency (Hz):', round(float(fmax), 3))

fig, axes = plt.subplots(2, 1, figsize=(8.8, 5.2), sharex=True)
axes[0].plot(t_ms/1000.0, exc_drive, label='Exc + input', lw=1.4)
axes[0].plot(t_ms/1000.0, inh, label='Inh', lw=1.2)
axes[0].legend(frameon=False)
axes[0].set_ylabel('Rate (a.u.)')
axes[0].set_title('Single-region utility demo')
axes[0].grid(alpha=0.2)

axes[1].plot(frq, pexc, label='Exc PSD', lw=1.4)
axes[1].plot(frq, pinh, label='Inh PSD', lw=1.2)
axes[1].set_xlim(0, 40)
axes[1].set_xlabel('Frequency (Hz)')
axes[1].set_ylabel('Power')
axes[1].legend(frameon=False)
axes[1].grid(alpha=0.2)
fig.tight_layout()
fig.savefig(fig_dir / '06_single_region_utility_demo.svg', bbox_inches='tight')
fig.savefig(fig_dir / '06_single_region_utility_demo.pdf', bbox_inches='tight')
plt.show()

## Output

Figures are written to:
`notebooks/outputs/repro_maria_sacha_nature/figures/paper_repro/`.